In [11]:
%pip install -q langchain langchain-groq requests

In [2]:
import os
import requests

from langchain_groq import ChatGroq
from langchain.tools import tool

In [7]:
url = f"https://v6.exchangerate-api.com/v6/{EXCHANGE_RATE_API_KEY}/latest/USD"

response = requests.get(url)

data = response.json()

print(data["result"])
print(data["conversion_rates"]["INR"])

success
92.753


In [9]:
def get_exchange_rate(base, target):

    url = f"https://v6.exchangerate-api.com/v6/{EXCHANGE_RATE_API_KEY}/latest/{base}"

    response = requests.get(url)
    data = response.json()

    rate = data["conversion_rates"][target]

    return rate

In [12]:
@tool
def get_exchange_rate(base: str, target: str) -> float:
    """Get the latest exchange rate between two currencies."""

    url = f"https://v6.exchangerate-api.com/v6/{EXCHANGE_RATE_API_KEY}/latest/{base}"

    response = requests.get(url)
    data = response.json()

    return data["conversion_rates"][target]

In [13]:
print(get_exchange_rate.name)
print(get_exchange_rate.description)

get_exchange_rate
Get the latest exchange rate between two currencies.


In [ ]:
@tool
def convert_currency(amount: float, base: str, target: str) -> float:
    """Convert an amount from one currency to another."""

    rate = get_exchange_rate.invoke({
        "base": base,
        "target": target
    })

    return amount * rate

In [15]:
result = convert_currency.invoke({
    "amount": 100,
    "base_currency": "USD",
    "target_currency": "INR"
})

print(result)

100.00 USD is equivalent to 9275.30 INR at an exchange rate of 92.7530.


In [16]:
currency_toolkit = [
    get_exchange_rate,
    convert_currency
]

print(currency_toolkit)

[StructuredTool(name='get_exchange_rate', description='Get the latest exchange rate between two currencies.', args_schema=<class 'langchain_core.utils.pydantic.get_exchange_rate'>, func=<function get_exchange_rate at 0x7b15afbd36a0>), StructuredTool(name='convert_currency', description='Converts an amount of currency from a base currency to a target currency.\n\nArgs:\n    amount: The numeric value to convert. Must be a positive number.\n    base_currency: The three-letter ISO currency code of the source currency (e.g. USD, EUR).\n    target_currency: The three-letter ISO currency code of the destination currency (e.g. INR, GBP).', args_schema=<class 'langchain_core.utils.pydantic.convert_currency'>, func=<function convert_currency at 0x7b15adaacae0>)]


In [17]:
llm = ChatGroq(
    model="openai/gpt-oss-20b",
    api_key=GROQ_API_KEY,
    temperature=0
)

print("Groq LLM connected!")

Groq LLM connected!


In [18]:
llm_with_tools = llm.bind_tools(currency_toolkit)

print("Tools successfully bind ho gaye!")

Tools successfully bind ho gaye!


In [19]:
response = llm_with_tools.invoke(
    "Convert 100 USD to INR"
)

print(response.tool_calls)

[{'name': 'convert_currency', 'args': {'amount': 100, 'base_currency': 'USD', 'target_currency': 'INR'}, 'id': 'fc_5ef311c0-8fe4-46be-a884-eee0e3217872', 'type': 'tool_call'}]


In [20]:
tool_call = response.tool_calls[0]

selected_tool = {
    "get_exchange_rate": get_exchange_rate,
    "convert_currency": convert_currency
}[tool_call["name"]]

result = selected_tool.invoke(tool_call["args"])

print("Tool:", tool_call["name"])
print("Result:", result)

Tool: convert_currency
Result: 100.00 USD is equivalent to 9275.30 INR at an exchange rate of 92.7530.


In [21]:
from langchain_core.messages import HumanMessage, ToolMessage

In [22]:
messages = [
    HumanMessage(content="Convert 100 USD to INR"),
    response,
    ToolMessage(
        content=str(result),
        tool_call_id=tool_call["id"]
    )
]

final_response = llm_with_tools.invoke(messages)

print(final_response.content)

Sure!  
**100 USD ≈ 9,275.30 INR**  
(Using an exchange rate of 92.7530 INR per USD.)


In [24]:
def ask_currency_mate(question):

    response = llm_with_tools.invoke(question)

    if not response.tool_calls:
        return response.content

    tool_call = response.tool_calls[0]

    selected_tool = {
        "get_exchange_rate": get_exchange_rate,
        "convert_currency": convert_currency
    }[tool_call["name"]]

    result = selected_tool.invoke(tool_call["args"])

    messages = [
        HumanMessage(content=question),
        response,
        ToolMessage(
            content=str(result),
            tool_call_id=tool_call["id"]
        )
    ]

    final_response = llm_with_tools.invoke(messages)

    return final_response.content

In [25]:
print(ask_currency_mate("Convert 500 EUR to INR"))

**500 EUR → INR**

Using the current exchange rate of **1 EUR = 101.6586 INR**:

\[
500 \text{ EUR} \times 101.6586 \frac{\text{INR}}{\text{EUR}} = 50{,}829.30 \text{ INR}
\]

So, 500 EUR is approximately **₹50,829.30**.


In [26]:
print(ask_currency_mate("Convert 100 USD to INR"))
print(ask_currency_mate("Get the exchange rate of GBP to INR"))

Sure!  
**100 USD ≈ 9,275.30 INR**  
(Using an exchange rate of 92.7530 INR per USD.)
**Exchange Rate (GBP → INR):** 1 GBP = **121.3229 INR**


In [27]:
print(ask_currency_mate("What is CurrencyMate AI and what can you do?"))

**CurrencyMate AI** is a conversational AI assistant built to help you navigate the world of foreign exchange and international finance. Think of it as a pocket‑size financial concierge that can:

| What it can do | How it helps you |
|----------------|------------------|
| **Instant currency conversion** | Convert any amount from one currency to another on the fly. |
| **Real‑time exchange rates** | Pull the latest rates for any currency pair, so you always know the current value. |
| **Historical rate lookup** | See how a currency has moved over days, weeks, or months. |
| **Currency trend insights** | Get quick summaries of recent trends, volatility, and key drivers. |
| **Travel budgeting** | Estimate how much you’ll need in local currency for trips, flights, hotels, etc. |
| **Business finance** | Help with invoicing, pricing, and cost‑analysis across multiple currencies. |
| **Educational snippets** | Explain forex concepts, market mechanics, and how rates are set. |
| **Alerts &

In [28]:
print(ask_currency_mate("What is the current exchange rate of GBP to INR?"))

The current exchange rate is **1 GBP ≈ 121.32 INR** (as retrieved from the latest data).


# CurrencyMate AI

## Project Overview
CurrencyMate AI is a beginner-level LLM application that answers currency-related questions using LangChain tools and a Groq-hosted LLM.

## Tech Stack
- Python
- Google Colab
- LangChain
- Groq LLM (`openai/gpt-oss-20b`)
- ExchangeRate-API
- Requests

## Key Concepts Learned
1. Creating custom tools using `@tool`
2. Creating a toolkit
3. Binding tools to an LLM using `bind_tools()`
4. Understanding LLM-generated tool calls
5. Executing tools using `.invoke()`
6. Returning tool results using `ToolMessage`
7. Building a reusable function to handle user questions

## Available Tools
- `get_exchange_rate`: Fetches the exchange rate between two currencies.
- `convert_currency`: Converts an amount from one currency to another.

## Limitations
- Basic error handling.
- Handles only one tool call per request.
- Uses a free-tier external exchange-rate API.
- Intended for learning, not financial decisions.
